# Dual-Mamba Pipeline — Colab A100 학습 드라이버

**준비**:
1. 이 폴더 (`dual_mamba/`) 통째로 Google Drive `/MyDrive/Colab Notebooks/` 아래 업로드.
2. Colab 런타임을 **A100** 으로 변경 (Runtime → Change runtime type → A100 GPU).
3. 셀 순서대로 실행.

**구조**:
- Stage 1 (Mamba A): past 시계열 + future tbill 시나리오 → future excess_liq
- Stage 1.5 (Bridge): 26w rolling sum (deterministic, 미분 가능)
- Stage 2 (Mamba B): past sp_return + future cumulative → future sp_return
- 학습 모드: teacher_forcing (Bridge 입력에 관측 future 사용)

## 1. Drive mount + 작업 디렉토리 이동

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
WORKDIR = '/content/drive/MyDrive/Colab Notebooks/dual_mamba'
os.chdir(WORKDIR)
print('cwd:', os.getcwd())
print('files:', sorted(os.listdir('.')))

## 2. 의존성 설치 (mambapy)

In [ ]:
!pip install -q mambapy

## 3. GPU + import 확인

In [ ]:
import torch
print('CUDA available :', torch.cuda.is_available())
if torch.cuda.is_available():
    print('Device         :', torch.cuda.get_device_name(0))
    print('VRAM total     :', f'{torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')

from mambapy.mamba import Mamba, MambaConfig
print('mambapy import OK')

## 4. Smoke test (모델 + 데이터)

In [ ]:
!python data_loader.py

In [ ]:
!python dual_mamba.py

## 5. 본 학습 (A100, 약 30~45분)

기본: K=2, d_model=64, n_layers=2, params 약 670K, batch=32, max_epochs=60, patience=15.

In [ ]:
!python train.py \
    --tag dualmamba_K2_d64_l2 \
    --K 2 --d-model 64 --n-layers 2 \
    --lr 5e-4 --batch 32 --max-epochs 60 --patience 15 \
    --lambda-nll2 1.0 --val-fraction 0.15 \
    --train-csv data/weekly_ppbond_train.csv \
    --test-csv  data/weekly_ppbond_test.csv \
    --out-dir   result \
    --seed 42 \
    2>&1 | tee result/dualmamba_K2_d64_l2_train.log

## 6. 평가 (시나리오 생성 + marginal/PIT 진단)

n_samples=100 윈도우당 미래 시나리오 100개 생성, max_windows=50 처음 50 윈도우만 평가 (시간 절약).
전부 평가하려면 `--max-windows 0` 또는 큰 값.

In [ ]:
!python eval.py \
    --ckpt result/dualmamba_K2_d64_l2_best.pt \
    --test-csv data/weekly_ppbond_test.csv \
    --n-samples 100 --max-windows 50 \
    --out-dir result \
    2>&1 | tee result/dualmamba_K2_d64_l2_eval.log

## 7. 결과 시각화 (간단)

In [ ]:
import json, pandas as pd
import matplotlib.pyplot as plt

log = pd.read_csv('result/dualmamba_K2_d64_l2_trainlog.csv')
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].plot(log['epoch'], log['tr_nll1'], label='train'); axes[0].plot(log['epoch'], log['va_nll1'], label='val')
axes[0].set_title('NLL_1 (Stage 1, excess_liq)'); axes[0].legend(); axes[0].grid(True)
axes[1].plot(log['epoch'], log['tr_nll2'], label='train'); axes[1].plot(log['epoch'], log['va_nll2'], label='val')
axes[1].set_title('NLL_2 (Stage 2, sp_return)'); axes[1].legend(); axes[1].grid(True)
axes[2].plot(log['epoch'], log['tr_total'], label='train'); axes[2].plot(log['epoch'], log['va_total'], label='val')
axes[2].set_title('NLL total'); axes[2].legend(); axes[2].grid(True)
plt.tight_layout(); plt.show()

with open('result/dualmamba_K2_d64_l2_summary.json') as f:
    summary = json.load(f)
print('train summary:', json.dumps({k: summary[k] for k in ['best_val_loss', 'best_epoch', 'test_metrics']}, indent=2))

with open('result/dualmamba_K2_d64_l2_eval_summary.json') as f:
    es = json.load(f)
print('eval summary:', json.dumps({k: es[k] for k in ['test_nll1', 'test_nll2', 'pit_z1', 'pit_z2',
    'marginal_excess_liq_obs', 'marginal_excess_liq_gen',
    'marginal_sp_return_obs', 'marginal_sp_return_gen']}, indent=2))